# Notebook 02 — Pipeline de Pré-processamento**Referência na monografia:** Seção 3.3 (Pipeline de Pré-processamento)**Etapas executadas, na ordem descrita na metodologia:**1. Tratamento de valores ausentes (§3.3.2)2. Codificação de variáveis categóricas (§3.3.3)3. Divisão treino/teste estratificada 80/20 (§3.3.5)4. Normalização das variáveis numéricas (§3.3.4)5. Balanceamento de classes com SMOTE (§3.3.6)> **Ordem crítica.** A divisão treino/teste ocorre **antes** da normalização e do SMOTE.> Inverter essa ordem produz vazamento de dados (*data leakage*) e infla artificialmente> as métricas — exatamente o erro que a Seção 3.3.6 se compromete a evitar.**Saída gerada:** `data/dados_processados.pkl`

In [ ]:
import osimport pickleimport warningsimport numpy as npimport pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom imblearn.over_sampling import SMOTEwarnings.filterwarnings('ignore')RANDOM_STATE = 42np.random.seed(RANDOM_STATE)BASE_DIR = os.path.dirname(os.getcwd())DATA_DIR = os.path.join(BASE_DIR, 'data')TAB_DIR = os.path.join(BASE_DIR, 'outputs', 'tabelas')os.makedirs(TAB_DIR, exist_ok=True)df = pd.read_csv(os.path.join(DATA_DIR, 'telco_churn.csv'))print(f'Base carregada: {df.shape[0]:,} registros, {df.shape[1]} variáveis')

## 1. Tratamento de valores ausentes (§3.3.2)Conversão de `TotalCharges` para tipo numérico e imputação por zero, justificada pelanatureza semântica dos registros afetados (clientes sem cobrança acumulada efetivada).

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')n_ausentes = df['TotalCharges'].isnull().sum()print(f'Registros com TotalCharges ausente: {n_ausentes}')if n_ausentes > 0:    tenure_afetados = df.loc[df['TotalCharges'].isnull(), 'tenure'].unique()    print(f'Valores de tenure nesses registros: {tenure_afetados}')    print('Confirma a hipótese semântica: são clientes sem cobrança acumulada.')df['TotalCharges'] = df['TotalCharges'].fillna(0)print(f'\nApós imputação por zero — nulos restantes: {df.isnull().sum().sum()}')

## 2. Preparação da variável-alvo e descarte do identificador

In [ ]:
y = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)X = df.drop(columns=['customerID', 'Churn'])print(f'Variável customerID descartada (sem informação preditiva).')print(f'Matriz de preditores: {X.shape[1]} variáveis')print(f'Distribuição do alvo: {y.value_counts().to_dict()}')

## 3. Codificação de variáveis categóricas (§3.3.3)Duas estratégias, conforme a cardinalidade de cada variável:- **Label Encoding** para variáveis binárias- **One-Hot Encoding** para variáveis com três ou mais categoriasO parâmetro `drop_first=True` no One-Hot evita multicolinearidade perfeita (*dummy trap*),relevante para a Regressão Logística.

In [ ]:
categoricas = X.select_dtypes(include='object').columns.tolist()binarias = [c for c in categoricas if X[c].nunique() == 2]multiclasse = [c for c in categoricas if X[c].nunique() > 2]print(f'Binárias (Label Encoding) — {len(binarias)}:')print(f'  {binarias}')print(f'\nMulticlasse (One-Hot Encoding) — {len(multiclasse)}:')for c in multiclasse:    print(f'  {c}: {sorted(X[c].unique())}')

In [ ]:
# Label Encoding para variáveis bináriasmapa_binario = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}for col in binarias:    X[col] = X[col].map(mapa_binario).astype(int)# One-Hot Encoding para variáveis multiclasseX = pd.get_dummies(X, columns=multiclasse, drop_first=True, dtype=int)print(f'Dimensão após codificação: {X.shape[1]} variáveis')print(f'\nVariáveis resultantes:')for i, col in enumerate(X.columns, 1):    print(f'  {i:>2}. {col}')

## 4. Divisão entre treino e teste (§3.3.5)Proporção 80/20 com estratificação pela variável-alvo e semente fixa em 42.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(    X, y,    test_size=0.20,    stratify=y,    random_state=RANDOM_STATE)print(f'Conjunto de treino: {X_treino.shape[0]:,} registros')print(f'Conjunto de teste:  {X_teste.shape[0]:,} registros')print()print(f'Taxa de evasão no treino: {y_treino.mean() * 100:.2f}%')print(f'Taxa de evasão no teste:  {y_teste.mean() * 100:.2f}%')print('\nEstratificação preservou a proporção original em ambos os conjuntos.')

## 5. Normalização das variáveis numéricas (§3.3.4)O `StandardScaler` é ajustado **exclusivamente** sobre o conjunto de treino. Os parâmetrosestimados (média e desvio padrão) são então aplicados ao conjunto de teste, evitando queinformação do teste vaze para o treinamento.

In [ ]:
NUMERICAS = ['tenure', 'MonthlyCharges', 'TotalCharges']scaler = StandardScaler()X_treino_esc = X_treino.copy()X_teste_esc = X_teste.copy()# fit_transform apenas no treinoX_treino_esc[NUMERICAS] = scaler.fit_transform(X_treino[NUMERICAS])# transform (sem fit) no testeX_teste_esc[NUMERICAS] = scaler.transform(X_teste[NUMERICAS])print('Parâmetros estimados no conjunto de treino:')params = pd.DataFrame({    'Variável': NUMERICAS,    'Média': scaler.mean_.round(3),    'Desvio padrão': np.sqrt(scaler.var_).round(3)})display(params)print('\nVerificação no treino (esperado: média ~0, desvio ~1):')display(X_treino_esc[NUMERICAS].describe().loc[['mean', 'std']].round(4))

## 6. Balanceamento de classes com SMOTE (§3.3.6)Aplicado **somente ao conjunto de treino**, com `k_neighbors=5` e `sampling_strategy=1.0`(balanceamento total). O conjunto de teste permanece em sua distribuição originaldesbalanceada — condição para que as métricas finais reflitam o desempenho esperado emprodução.

In [ ]:
smote = SMOTE(    sampling_strategy=1.0,    k_neighbors=5,    random_state=RANDOM_STATE)X_treino_bal, y_treino_bal = smote.fit_resample(X_treino_esc, y_treino)print('ANTES do SMOTE (treino):')print(f'  Não-evasores: {(y_treino == 0).sum():,}')print(f'  Evasores:     {(y_treino == 1).sum():,}')print(f'  Total:        {len(y_treino):,}')print('\nDEPOIS do SMOTE (treino):')print(f'  Não-evasores: {(y_treino_bal == 0).sum():,}')print(f'  Evasores:     {(y_treino_bal == 1).sum():,}')print(f'  Total:        {len(y_treino_bal):,}')print(f'  Exemplos sintéticos gerados: {len(y_treino_bal) - len(y_treino):,}')print('\nCONJUNTO DE TESTE (intocado):')print(f'  Não-evasores: {(y_teste == 0).sum():,}')print(f'  Evasores:     {(y_teste == 1).sum():,}')print(f'  Taxa de evasão preservada: {y_teste.mean() * 100:.2f}%')

## 7. Persistência dos dados processadosTodos os objetos necessários às etapas seguintes são serializados em um único arquivo,garantindo que os Notebooks 03 e 04 operem exatamente sobre os mesmos dados.

In [ ]:
dados = {    # Treino balanceado — usado para treinar os modelos    'X_treino_bal': X_treino_bal,    'y_treino_bal': y_treino_bal,    # Treino original (sem SMOTE) — referência e background do SHAP    'X_treino_esc': X_treino_esc,    'y_treino': y_treino,    # Teste — distribuição original, usado apenas na avaliação final    'X_teste_esc': X_teste_esc,    'y_teste': y_teste,    # Metadados    'colunas': X.columns.tolist(),    'numericas': NUMERICAS,    'scaler': scaler,    'random_state': RANDOM_STATE,}caminho = os.path.join(DATA_DIR, 'dados_processados.pkl')with open(caminho, 'wb') as f:    pickle.dump(dados, f)print(f'Dados processados salvos em: {caminho}')# Tabela-resumo do pipeline, aproveitável na monografiaresumo = pd.DataFrame([    {'Etapa': 'Base original', 'Registros': len(df), 'Variáveis': df.shape[1] - 1},    {'Etapa': 'Após codificação', 'Registros': len(X), 'Variáveis': X.shape[1]},    {'Etapa': 'Treino (80%)', 'Registros': len(X_treino), 'Variáveis': X.shape[1]},    {'Etapa': 'Treino após SMOTE', 'Registros': len(X_treino_bal), 'Variáveis': X.shape[1]},    {'Etapa': 'Teste (20%)', 'Registros': len(X_teste), 'Variáveis': X.shape[1]},])display(resumo)resumo.to_csv(os.path.join(TAB_DIR, 'tab_resumo_preprocessamento.csv'), index=False)print('\nPróxima etapa: Notebook 03 — Modelagem e Validação')